# Notebook 3 — Fused MLP Kernel (Partial Fusion)

**Goal**: reduce kernel-launch overhead by fusing `gate_proj + up_proj + silu(gate) * up` into a single Triton kernel. `down_proj` stays as a separate block-sparse linear. So each MLP layer goes from 3 kernel launches to 2.

The fused kernel does two block-sparse matmuls inside a single grid, then applies `silu(gate_acc) * up_acc` elementwise before writing the output — so the intermediate `gate` tensor never round-trips VRAM.

We reuse the bitmap architecture (dense weight + bitmap, hybrid M<64 fallback) because (a) BCSR didn't pan out and (b) the bitmap kernel was actually our best design point.

Comparing 5 configs: Dense, Bitmap-BlockSparse 21%, Bitmap-BlockSparse 50%, **Fused 21%**, **Fused 50%**.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import triton
import triton.language as tl
import matplotlib.pyplot as plt
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
import time, gc, re, types

import config

torch.set_float32_matmul_precision("high")
plt.rcParams["figure.dpi"] = 120
print(f"torch: {torch.__version__}  triton: {triton.__version__}")
print(f"device: {torch.cuda.get_device_name(0)}")

TILE_R, TILE_C = config.TILE_SIZES[0]
print(f"tile size: ({TILE_R}, {TILE_C})")

torch: 2.6.0+cu124  triton: 3.2.0
device: NVIDIA GeForce RTX 4070 Laptop GPU
tile size: (128, 128)


## 1. Load model

In [2]:
print(f"Loading {config.MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()
dtype = getattr(torch, config.TORCH_DTYPE)
print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

Loading google/gemma-3-1b-it...
params: 1.00B


## 2. ActW calibration + build masks (21% effective and 50% uniform)

In [3]:
from datasets import load_dataset
from tqdm import tqdm

def is_mlp(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def compute_tile_frob(W, tr=TILE_R, tc=TILE_C):
    W = W.detach().float().cpu()
    out_dim, in_dim = W.shape
    nr, nc = out_dim // tr, in_dim // tc
    Wt = W[:nr*tr, :nc*tc].reshape(nr, tr, nc, tc).permute(0, 2, 1, 3)
    return Wt.reshape(nr, nc, -1).norm(dim=-1).numpy()

# --- Load calibration data ---
print("Loading calibration data...")
cal_ds = load_dataset(
    config.CALIBRATION_DATASET,
    config.CALIBRATION_SUBSET,
    split="train",
    streaming=True,
    trust_remote_code=True,
)
cal_texts = []
for i, ex in enumerate(cal_ds):
    if i >= config.CALIBRATION_SAMPLES:
        break
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
cal_encodings = [
    tokenizer(t, return_tensors="pt", max_length=config.CALIBRATION_SEQ_LEN, truncation=True)
    for t in cal_texts
]
print(f"loaded {len(cal_encodings)} samples")

Loading calibration data...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

loaded 1022 samples


In [4]:
# Frobenius
importance_frob = {}
for name, param in model.named_parameters():
    if param.ndim != 2 or not is_mlp(name):
        continue
    importance_frob[name] = compute_tile_frob(param)
print(f"frob scored: {len(importance_frob)} matrices")

# Activation hooks
activation_stats = {}
hooks = []
def make_hook(pname, in_dim):
    nc = in_dim // TILE_C
    def h(mod, inp, out):
        x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
        x = x[:, :nc*TILE_C].reshape(x.shape[0], nc, TILE_C)
        cn = x.norm(dim=-1).mean(dim=0).cpu()
        if pname not in activation_stats:
            activation_stats[pname] = (cn, 1)
        else:
            s, c = activation_stats[pname]
            activation_stats[pname] = (s + cn, c + 1)
    return h

for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    pn = name + ".weight"
    if not is_mlp(pn) or module.in_features < TILE_C:
        continue
    hooks.append(module.register_forward_hook(make_hook(pn, module.in_features)))

print(f"calibrating on {len(cal_encodings)} samples (forward only)...")
with torch.no_grad():
    for enc in tqdm(cal_encodings, desc="calib"):
        model(**{k: v.to(config.DEVICE) for k, v in enc.items()})
for h in hooks:
    h.remove()
print(f"activation stats collected for {len(activation_stats)} matrices")

frob scored: 78 matrices
calibrating on 1022 samples (forward only)...


calib: 100%|█████████████████████████████████████████████████████| 1022/1022 [00:53<00:00, 18.99it/s]

activation stats collected for 78 matrices


In [5]:
# Activation-weighted importance + per-component-type z-score
importance_actw = {}
for name, fmap in importance_frob.items():
    if name not in activation_stats:
        continue
    cn, count = activation_stats[name]
    importance_actw[name] = fmap * (cn / count).numpy()[np.newaxis, :]

comp_stats = {}
for ct in config.PRUNE_TARGETS_PATTERNS:
    vals = np.concatenate([m.ravel() for n, m in importance_actw.items() if ct in n])
    comp_stats[ct] = (vals.mean(), vals.std())

importance_actw_norm = {}
for name, m in importance_actw.items():
    mu, sd = comp_stats[get_component_type(name)]
    importance_actw_norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)

# Masks at 30% target with 50% per-matrix cap => 21% effective
PRUNE_RATIO = 0.30
all_scores = np.concatenate([m.ravel() for m in importance_actw_norm.values()])
threshold = float(np.percentile(all_scores, PRUNE_RATIO * 100))
def capped_mask(norm_map, thresh, max_prune=config.MAX_PRUNE_PER_MATRIX):
    gm = norm_map < thresh
    if gm.mean() <= max_prune:
        return gm
    local = float(np.percentile(norm_map.ravel(), max_prune * 100))
    return norm_map < local
prune_masks_21 = {name: capped_mask(m, threshold) for name, m in importance_actw_norm.items()}

# Masks at 50% uniform per matrix
prune_masks_50 = {}
for name, norm_map in importance_actw_norm.items():
    local = float(np.percentile(norm_map.ravel(), 50))
    prune_masks_50[name] = norm_map < local

def eff_sparsity(masks):
    total = sum(m.size for m in masks.values())
    pruned = sum(m.sum() for m in masks.values())
    return pruned, total, pruned / total

p21, t21, r21 = eff_sparsity(prune_masks_21)
p50, t50, r50 = eff_sparsity(prune_masks_50)
print(f"21% ActW (capped): {p21}/{t21} = {100*r21:.1f}% effective")
print(f"50% uniform:       {p50}/{t50} = {100*r50:.1f}% effective")

21% ActW (capped): 7966/37908 = 21.0% effective
50% uniform:       18954/37908 = 50.0% effective


## 3. Bitmap block-sparse kernel + `BlockSparseLinear` (with hybrid dispatch)

Same design as notebook 2. Reproduced here so this notebook is self-contained.

In [6]:
@triton.jit
def block_sparse_matmul_kernel(
    X_ptr, W_ptr, Y_ptr, bitmap_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wn, stride_wk,
    stride_ym, stride_yn,
    stride_bn, stride_bk,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    x_ptrs = X_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
    w_ptrs = W_ptr + offs_n[:, None] * stride_wn + offs_k[None, :] * stride_wk
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    n_k_tiles = tl.cdiv(K, BLOCK_K)
    for k_tile in range(n_k_tiles):
        nz = tl.load(bitmap_ptr + pid_n * stride_bn + k_tile * stride_bk)
        if nz != 0:
            k_start = k_tile * BLOCK_K
            x_mask = (offs_m[:, None] < M) & ((offs_k[None, :] + k_start) < K)
            w_mask = (offs_n[:, None] < N) & ((offs_k[None, :] + k_start) < K)
            x = tl.load(x_ptrs, mask=x_mask, other=0.0)
            w = tl.load(w_ptrs, mask=w_mask, other=0.0)
            acc += tl.dot(x, tl.trans(w))
        x_ptrs += BLOCK_K * stride_xk
        w_ptrs += BLOCK_K * stride_wk
    y_ptrs = Y_ptr + offs_m[:, None] * stride_ym + offs_n[None, :] * stride_yn
    y_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(y_ptrs, acc.to(Y_ptr.dtype.element_ty), mask=y_mask)


def block_sparse_matmul(x, w, bitmap, block_m=64, block_n=TILE_R, block_k=TILE_C):
    M, K = x.shape
    N, _ = w.shape
    y = torch.empty((M, N), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    block_sparse_matmul_kernel[grid](
        x, w, y, bitmap,
        M, N, K,
        x.stride(0), x.stride(1),
        w.stride(0), w.stride(1),
        y.stride(0), y.stride(1),
        bitmap.stride(0), bitmap.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
    )
    return y


class BlockSparseLinear(nn.Module):
    SMALL_M_THRESHOLD = 64

    def __init__(self, in_features, out_features, bitmap, weight, bias=None,
                 tile_r=TILE_R, tile_c=TILE_C, small_m_threshold=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.tile_r = tile_r
        self.tile_c = tile_c
        self.small_m_threshold = (small_m_threshold if small_m_threshold is not None
                                  else self.SMALL_M_THRESHOLD)
        self.register_buffer("weight", weight.contiguous())
        self.register_buffer("bitmap", bitmap.contiguous())
        if bias is not None:
            self.register_buffer("bias", bias.contiguous())
        else:
            self.bias = None

    def forward(self, x):
        orig_shape = x.shape
        x2 = x.reshape(-1, orig_shape[-1])
        M = x2.shape[0]
        if M < self.small_m_threshold:
            y = x2 @ self.weight.T
        else:
            if not x2.is_contiguous():
                x2 = x2.contiguous()
            y = block_sparse_matmul(x2, self.weight, self.bitmap,
                                    block_n=self.tile_r, block_k=self.tile_c)
        if self.bias is not None:
            y = y + self.bias
        return y.reshape(*orig_shape[:-1], self.out_features)

    @classmethod
    def from_dense(cls, linear, prune_mask, tile_r=TILE_R, tile_c=TILE_C):
        out_f, in_f = linear.weight.shape
        nr, nc = out_f // tile_r, in_f // tile_c
        keep = (~prune_mask[:nr, :nc]).astype(np.int8)
        bitmap = torch.from_numpy(keep).to(linear.weight.device)
        w = linear.weight.detach().clone()
        for i in range(nr):
            for j in range(nc):
                if prune_mask[i, j]:
                    w[i*tile_r:(i+1)*tile_r, j*tile_c:(j+1)*tile_c] = 0
        b = linear.bias.detach().clone() if linear.bias is not None else None
        return cls(in_f, out_f, bitmap, w, b, tile_r, tile_c)

print("BlockSparseLinear (with hybrid dispatch) ready.")

BlockSparseLinear (with hybrid dispatch) ready.


## 4. Fused kernel: `gate_proj + up_proj + silu * up` in one shot

The kernel walks the K-dimension once, accumulating two separate tile-sparse matmuls (gate and up). At the end, it applies `silu(gate_acc) * up_acc` elementwise on the FP32 accumulators and stores one output tensor.

Memory: reads `x` once per K-tile (instead of twice — once per linear). Writes `h` once (instead of writing gate + up separately and then reading both for the elementwise op). Also one fewer kernel launch per layer.

At each K-tile, we check both bitmaps. If either is non-zero we load `x` (shared between gate and up). If only gate is non-zero we skip the up load; if only up, skip gate.

In [7]:
@triton.jit
def fused_gate_up_silu_kernel(
    X_ptr, Wg_ptr, Wu_ptr, Bg_ptr, Bu_ptr, H_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wgn, stride_wgk,
    stride_wun, stride_wuk,
    stride_hm, stride_hn,
    stride_bgn, stride_bgk,
    stride_bun, stride_buk,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    x_ptrs  = X_ptr  + offs_m[:, None] * stride_xm  + offs_k[None, :] * stride_xk
    wg_ptrs = Wg_ptr + offs_n[:, None] * stride_wgn + offs_k[None, :] * stride_wgk
    wu_ptrs = Wu_ptr + offs_n[:, None] * stride_wun + offs_k[None, :] * stride_wuk

    acc_g = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    acc_u = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    n_k_tiles = tl.cdiv(K, BLOCK_K)
    for k_tile in range(n_k_tiles):
        nz_g = tl.load(Bg_ptr + pid_n * stride_bgn + k_tile * stride_bgk)
        nz_u = tl.load(Bu_ptr + pid_n * stride_bun + k_tile * stride_buk)
        if (nz_g != 0) | (nz_u != 0):
            k_start = k_tile * BLOCK_K
            x_mask = (offs_m[:, None] < M) & ((offs_k[None, :] + k_start) < K)
            x = tl.load(x_ptrs, mask=x_mask, other=0.0)
            w_n_mask_k = (offs_n[:, None] < N) & ((offs_k[None, :] + k_start) < K)
            if nz_g != 0:
                wg = tl.load(wg_ptrs, mask=w_n_mask_k, other=0.0)
                acc_g += tl.dot(x, tl.trans(wg))
            if nz_u != 0:
                wu = tl.load(wu_ptrs, mask=w_n_mask_k, other=0.0)
                acc_u += tl.dot(x, tl.trans(wu))
        x_ptrs  += BLOCK_K * stride_xk
        wg_ptrs += BLOCK_K * stride_wgk
        wu_ptrs += BLOCK_K * stride_wuk

    # silu(g) * u  where silu(g) = g * sigmoid(g)
    silu_g = acc_g * tl.sigmoid(acc_g)
    h = silu_g * acc_u

    h_ptrs = H_ptr + offs_m[:, None] * stride_hm + offs_n[None, :] * stride_hn
    h_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(h_ptrs, h.to(H_ptr.dtype.element_ty), mask=h_mask)


def fused_gate_up_silu(x, w_gate, w_up, bitmap_gate, bitmap_up,
                       block_m=64, block_n=TILE_R, block_k=TILE_C):
    """x: (M, K), w_gate/up: (N, K) -> h: (M, N) = silu(x @ w_gate.T) * (x @ w_up.T)"""
    assert x.shape[1] == w_gate.shape[1] == w_up.shape[1]
    assert w_gate.shape == w_up.shape
    M, K = x.shape
    N, _ = w_gate.shape
    h = torch.empty((M, N), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    fused_gate_up_silu_kernel[grid](
        x, w_gate, w_up, bitmap_gate, bitmap_up, h,
        M, N, K,
        x.stride(0), x.stride(1),
        w_gate.stride(0), w_gate.stride(1),
        w_up.stride(0), w_up.stride(1),
        h.stride(0), h.stride(1),
        bitmap_gate.stride(0), bitmap_gate.stride(1),
        bitmap_up.stride(0), bitmap_up.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
    )
    return h

print("fused_gate_up_silu kernel ready (first call triggers compile).")

fused_gate_up_silu kernel ready (first call triggers compile).


In [8]:
# ---- Correctness: fused kernel vs unfused silu(x@W_g.T) * (x@W_u.T) on dense-pruned weights ----
torch.manual_seed(0)
Mt, Nt, Kt = 256, 1024, 1152
xt = torch.randn(Mt, Kt, device="cuda", dtype=dtype)
wgt = torch.randn(Nt, Kt, device="cuda", dtype=dtype) * 0.1
wut = torch.randn(Nt, Kt, device="cuda", dtype=dtype) * 0.1

pm_g = (np.random.rand(Nt // TILE_R, Kt // TILE_C) > 0.70).astype(np.int8)  # keep mask, 30% dropped
pm_u = (np.random.rand(Nt // TILE_R, Kt // TILE_C) > 0.70).astype(np.int8)
bg = torch.from_numpy(pm_g).to("cuda").contiguous()
bu = torch.from_numpy(pm_u).to("cuda").contiguous()

# Zero out the "pruned" tiles in reference weights
wg_zero = wgt.clone()
wu_zero = wut.clone()
for i in range(Nt // TILE_R):
    for j in range(Kt // TILE_C):
        if pm_g[i, j] == 0:
            wg_zero[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
        if pm_u[i, j] == 0:
            wu_zero[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0

# Unfused reference
gate_ref = xt @ wg_zero.T
up_ref   = xt @ wu_zero.T
h_ref = F.silu(gate_ref) * up_ref

# Fused — kernel uses same zeroed weights (kernel also skips zero-tiles via bitmap)
h_fused = fused_gate_up_silu(xt, wg_zero, wu_zero, bg, bu)

diff = (h_ref - h_fused).abs()
print(f"max abs diff: {diff.max().item():.5f}")
print(f"mean abs diff: {diff.mean().item():.6f}")
print(f"|h_ref| mean:  {h_ref.abs().mean().item():.4f}")
# Loose tolerance because fused accumulator stays in FP32 throughout, unfused casts intermediate to bf16
assert diff.max().item() < 0.5, f"fused kernel mismatch: max diff {diff.max().item()}"
print("fused kernel correctness: OK")

max abs diff: 0.25000
mean abs diff: 0.002853
|h_ref| mean:  1.1484
fused kernel correctness: OK


## 5. Apply to model — monkey-patch MLP forward

Gemma's MLP forward is `down_proj(silu(gate_proj(x)) * up_proj(x))`. We:

1. Convert `gate_proj`, `up_proj`, `down_proj` to `BlockSparseLinear` (so the dense-fallback path at M<64 still works correctly).
2. Monkey-patch the MLP module's `forward` to use `fused_gate_up_silu` when M ≥ threshold, falling through to the non-fused path otherwise.

With hybrid dispatch inside `BlockSparseLinear` + fused kernel here, the dispatch table is:
- M < 64 (decode): 3 dense matmul + silu/mul = same as dense baseline
- M ≥ 64 (prefill): fused kernel + `down_proj` block-sparse kernel = 2 kernel launches per layer (was 3)

In [9]:
FUSE_M_THRESHOLD = 64

def fused_mlp_forward(self, x):
    orig_shape = x.shape
    x2 = x.reshape(-1, orig_shape[-1])
    M = x2.shape[0]
    if M < FUSE_M_THRESHOLD:
        # Non-fused path — each BlockSparseLinear dispatches to dense internally for M<64
        gate = self.gate_proj(x)
        up   = self.up_proj(x)
        h = F.silu(gate) * up
        return self.down_proj(h)
    # Fused path
    if not x2.is_contiguous():
        x2 = x2.contiguous()
    h_flat = fused_gate_up_silu(
        x2,
        self.gate_proj.weight, self.up_proj.weight,
        self.gate_proj.bitmap, self.up_proj.bitmap,
    )
    h = h_flat.reshape(*orig_shape[:-1], h_flat.shape[-1])
    return self.down_proj(h)


def get_parent_and_attr(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]


def build_block_sparse_model(mask_dict, fuse_mlp=False):
    """Fresh model, swap MLP linears -> BlockSparseLinear, optionally monkey-patch MLP forward."""
    m = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        torch_dtype=getattr(torch, config.TORCH_DTYPE),
        device_map=config.DEVICE,
    )
    m.eval()
    # Swap Linears
    for name, module in list(m.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        pname = name + ".weight"
        if pname not in mask_dict:
            continue
        bsl = BlockSparseLinear.from_dense(module, mask_dict[pname]).to("cuda")
        parent, attr = get_parent_and_attr(m, name)
        setattr(parent, attr, bsl)
    # Optionally fuse
    if fuse_mlp:
        patched = 0
        for name, module in m.named_modules():
            if (hasattr(module, "gate_proj") and hasattr(module, "up_proj")
                and hasattr(module, "down_proj")
                and isinstance(module.gate_proj, BlockSparseLinear)):
                module.forward = types.MethodType(fused_mlp_forward, module)
                patched += 1
        print(f"  fused forward patched on {patched} MLP modules")
    torch.cuda.empty_cache()
    return m

print("build_block_sparse_model (with optional fusion) ready.")

build_block_sparse_model (with optional fusion) ready.


## 6. Benchmarks — 5 configs × 3 scenarios

Configs: Dense | Bitmap-BS 21% | Bitmap-BS 50% | **Fused 21%** | **Fused 50%**

Scenarios: short (decode-heavy) | long (mixed) | pure prefill

In [10]:
def bench_generate(model, inputs, max_new_tokens=50, n_warmup=2, n_iter=5):
    with torch.no_grad():
        for _ in range(n_warmup):
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iter

def bench_prefill(model, inputs, n_warmup=3, n_iter=10):
    with torch.no_grad():
        for _ in range(n_warmup):
            model(**inputs)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            model(**inputs)
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iter

# Build the two prompts
short_text = "Once upon a time"
long_text = " ".join(cal_texts[:10])
short_inputs = tokenizer(short_text, return_tensors="pt").to("cuda")
long_inputs  = tokenizer(long_text, return_tensors="pt", max_length=2000, truncation=True).to("cuda")
short_len = short_inputs["input_ids"].shape[1]
long_len  = long_inputs["input_ids"].shape[1]
print(f"short prompt: {short_len} tokens")
print(f"long prompt:  {long_len} tokens")

def bench_all(model, tag):
    s = bench_generate(model, short_inputs, max_new_tokens=50)
    l = bench_generate(model, long_inputs,  max_new_tokens=20)
    p = bench_prefill(model, long_inputs)
    print(f"[{tag:<20}] short: {s*1000:7.1f}ms | long: {l*1000:7.1f}ms | prefill: {p*1000:7.1f}ms | peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    return s, l, p

short prompt: 5 tokens
long prompt:  1953 tokens


In [11]:
# === Dense baseline ===
print("\n--- Dense ---")
del model  # free the one from section 1 (has hooks removed but still loaded)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

dense_model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
dense_model.eval()
d_short, d_long, d_prefill = bench_all(dense_model, "Dense")
del dense_model
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()


--- Dense ---


/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[Dense               ] short:  1413.7ms | long:   684.4ms | prefill:   182.8ms | peak GPU: 3.69 GB


In [12]:
# === Bitmap BlockSparse 21% (no fusion) ===
print("\n--- Bitmap-BS 21% ---")
m = build_block_sparse_model(prune_masks_21, fuse_mlp=False)
bs21_short, bs21_long, bs21_prefill = bench_all(m, "BS 21%")
del m
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# === Bitmap BlockSparse 50% ===
print("\n--- Bitmap-BS 50% ---")
m = build_block_sparse_model(prune_masks_50, fuse_mlp=False)
bs50_short, bs50_long, bs50_prefill = bench_all(m, "BS 50%")
del m
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()


--- Bitmap-BS 21% ---


/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[BS 21%              ] short:  1416.3ms | long:   712.2ms | prefill:   192.1ms | peak GPU: 3.87 GB

--- Bitmap-BS 50% ---
[BS 50%              ] short:  1418.1ms | long:   662.9ms | prefill:   159.8ms | peak GPU: 3.87 GB


In [13]:
# === Fused 21% ===
print("\n--- Fused 21% ---")
m = build_block_sparse_model(prune_masks_21, fuse_mlp=True)
f21_short, f21_long, f21_prefill = bench_all(m, "Fused 21%")
del m
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# === Fused 50% ===
print("\n--- Fused 50% ---")
m = build_block_sparse_model(prune_masks_50, fuse_mlp=True)
f50_short, f50_long, f50_prefill = bench_all(m, "Fused 50%")
del m
gc.collect(); torch.cuda.empty_cache()


--- Fused 21% ---
  fused forward patched on 26 MLP modules


/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/cmoryah/anaconda3/envs/chatbot/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[Fused 21%           ] short:  1506.8ms | long:   704.0ms | prefill:   188.4ms | peak GPU: 3.87 GB

--- Fused 50% ---
  fused forward patched on 26 MLP modules
[Fused 50%           ] short:  1512.0ms | long:   680.2ms | prefill:   159.0ms | peak GPU: 3.87 GB


In [14]:
# === Comparison table ===
print(f"\n{'='*110}")
print(f"{'Scenario':<34} {'Dense':>10} {'BS 21%':>10} {'BS 50%':>10} {'Fused 21%':>11} {'Fused 50%':>11}")
print("-" * 110)
def row(label, d, b21, b50, f21, f50):
    print(f"{label:<34} "
          f"{d*1000:>8.1f}ms {b21*1000:>8.1f}ms {b50*1000:>8.1f}ms "
          f"{f21*1000:>9.1f}ms {f50*1000:>9.1f}ms")
row("short + 50 gen (decode)",   d_short,   bs21_short,   bs50_short,   f21_short,   f50_short)
row("long + 20 gen (mixed)",     d_long,    bs21_long,    bs50_long,    f21_long,    f50_long)
row("prefill only",              d_prefill, bs21_prefill, bs50_prefill, f21_prefill, f50_prefill)
print("=" * 110)

# === Speedup vs dense ===
print(f"\n{'Scenario':<34} {'BS 21%':>9} {'BS 50%':>9} {'Fused 21%':>10} {'Fused 50%':>10}")
print("-" * 85)
def sp(label, d, b21, b50, f21, f50):
    print(f"{label:<34} {d/b21:>8.2f}x {d/b50:>8.2f}x {d/f21:>9.2f}x {d/f50:>9.2f}x")
sp("short + 50 gen",   d_short,   bs21_short,   bs50_short,   f21_short,   f50_short)
sp("long + 20 gen",    d_long,    bs21_long,    bs50_long,    f21_long,    f50_long)
sp("prefill only",     d_prefill, bs21_prefill, bs50_prefill, f21_prefill, f50_prefill)
print("=" * 85)

# === Fusion gain (Fused vs plain BS) ===
print(f"\nFusion gain (Fused / BS at same sparsity, >1.0x means fusion helped):")
print(f"{'Scenario':<34} {'Fused/BS 21%':>14} {'Fused/BS 50%':>14}")
print("-" * 70)
def fg(label, b21, f21, b50, f50):
    print(f"{label:<34} {b21/f21:>13.2f}x {b50/f50:>13.2f}x")
fg("short + 50 gen",   bs21_short,   f21_short,   bs50_short,   f50_short)
fg("long + 20 gen",    bs21_long,    f21_long,    bs50_long,    f50_long)
fg("prefill only",     bs21_prefill, f21_prefill, bs50_prefill, f50_prefill)
print("=" * 70)
print("Key test: Fused prefill > 1.0x over plain BS at both sparsities? -> fusion buys us something.")


Scenario                                Dense     BS 21%     BS 50%   Fused 21%   Fused 50%
--------------------------------------------------------------------------------------------------------------
short + 50 gen (decode)              1413.7ms   1416.3ms   1418.1ms    1506.8ms    1512.0ms
long + 20 gen (mixed)                 684.4ms    712.2ms    662.9ms     704.0ms     680.2ms
prefill only                          182.8ms    192.1ms    159.8ms     188.4ms     159.0ms

Scenario                              BS 21%    BS 50%  Fused 21%  Fused 50%
-------------------------------------------------------------------------------------
short + 50 gen                         1.00x     1.00x      0.94x      0.94x
long + 20 gen                          0.96x     1.03x      0.97x      1.01x
prefill only                           0.95x     1.14x      0.97x      1.15x

Fusion gain (Fused / BS at same sparsity, >1.0x means fusion helped):
Scenario                             Fused/BS 21%   Fu

## 7. MMLU correctness check on the fused 21% model

Quick MMLU eval to confirm the fused kernel preserves model behavior end-to-end.

In [15]:
from eval_mmlu import evaluate, print_results, save_results

m = build_block_sparse_model(prune_masks_21, fuse_mlp=True)
t0 = time.time()
results = evaluate(m, tokenizer, tag="fused_actw_30pct")
elapsed = time.time() - t0
results["meta"] = {
    "method": "actw",
    "prune_ratio": 0.30,
    "kernel": "fused_gate_up_silu + block_sparse_down",
    "eval_time_s": round(elapsed, 1),
}
print_results(results)
save_results(results)
del m
gc.collect(); torch.cuda.empty_cache()

  fused forward patched on 26 MLP modules


[fused_actw_30pct] MMLU subjects: 100%|██████████████████████████████| 10/10 [01:14<00:00,  7.46s/it]


  MMLU Results — fused_actw_30pct
  abstract_algebra               ████░░░░░░░░░░░░░░░░  21.0% (21/100)
  anatomy                        ████░░░░░░░░░░░░░░░░  23.0% (31/135)
  college_chemistry              ███████░░░░░░░░░░░░░  36.0% (36/100)
  college_computer_science       ██████░░░░░░░░░░░░░░  31.0% (31/100)
  econometrics                   ██████░░░░░░░░░░░░░░  30.7% (35/114)
  global_facts                   ███░░░░░░░░░░░░░░░░░  17.0% (17/100)
  machine_learning               ███░░░░░░░░░░░░░░░░░  17.0% (19/112)
  moral_scenarios                █████░░░░░░░░░░░░░░░  25.5% (228/895)
  professional_medicine          ████████░░░░░░░░░░░░  40.8% (111/272)
  us_foreign_policy              █████░░░░░░░░░░░░░░░  28.0% (28/100)
────────────────────────────────────────────────────────────
  OVERALL                                       27.5% (557/2028)

Results saved to results/mmlu_fused_actw_30pct.json


## Summary

Expected wins if fusion works:
- **Prefill**: launch overhead drops 33% (3 launches per layer → 2); intermediate `gate` never round-trips VRAM → memory traffic also drops. Should compound with the tile-skipping gain.
- **Decode (M<64)**: falls through to the same dense path as plain BS — no regression.
- **MMLU**: should match the ~28.4% we got on the plain BS 21% model.